# DeepQuality Mac 展示笔记本

本笔记本用于在 macOS 本机 Jupyter 环境中完整展示 DeepQuality 项目：数据读取、配置说明、S-DDAE 基线训练、SS-DDFAE 半监督训练、checkpoint 指标汇总和结果可视化。

主线实验使用 Debutanizer 数据集，并采用 `random_window` 样本划分、`seed=1`、`window_size=16`、`quality_delay=0`、`label_ratio=1.0`。该设置不引入 DCS，不新增第三类模型，只在 S-DDAE 与 SS-DDFAE 两个软测量模型内完成训练与评估。注意：`random_window` 用于同分布窗口样本泛化评估，不等同于严格按时间顺序的未来时间段外推。

## 1. Mac 运行前准备

请先在 macOS 终端中进入 DeepQuality 项目根目录，再启动 Jupyter Notebook 或 JupyterLab 打开本文件。

项目根目录应包含 `pyproject.toml`、`configs/`、`data/` 和 `src/deep_quality/`。

In [ ]:
from pathlib import Path
import os

def is_project_root(path):
    path = Path(path)
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'deep_quality').exists()

if not is_project_root(Path.cwd()):
    raise RuntimeError('当前目录不是 DeepQuality 项目根目录。请先切换到包含 pyproject.toml 和 src/deep_quality 的目录。')

print('当前项目目录:', Path.cwd())

## 2. 安装依赖

依赖统一写在 `pyproject.toml` 和 `requirements.txt` 中。本节只需要安装项目本身，`numpy`、`pandas`、`matplotlib`、`PyYAML` 和 `torch` 会随项目依赖一起安装。

如果当前 Mac 是 Apple Silicon 且 PyTorch 支持 MPS，本笔记本会优先用 MPS 做训练；否则使用 CPU。后处理、评估和在线推理默认使用 CPU，以保持兼容和可复现。

In [ ]:
%pip install -q -e .

In [ ]:
import sys
import torch

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('mps_built:', torch.backends.mps.is_built() if hasattr(torch.backends, 'mps') else False)
print('mps_available:', torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False)
print('recommended_training_device:', 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')

## 3. 项目结构检查

确认当前 macOS 环境中已经包含代码、配置、数据和说明文件。

In [ ]:
from pathlib import Path

files = sorted(path for path in Path('.').glob('**/*') if path.is_file())
for path in files[:80]:
    print(path.as_posix())

## 4. 数据集展示

默认数据集是 Debutanizer。输入变量为 `u1` 到 `u7`，目标质量变量为 `y`。

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/Debutanizer_Data.csv')
print('shape:', df.shape)
display(df.head())
display(df.describe().T)

## 5. 配置文件展示

`configs/base.yaml` 负责数据路径、划分比例、窗口长度、质量延迟和输出目录。模型配置文件覆盖基础配置中的同名字段。

In [ ]:
from pathlib import Path

for config_path in [
    'configs/base.yaml',
    'configs/sddae_random_window.yaml',
    'configs/ss_ddfae_random_window.yaml',
]:
    print()
    print('=' * 80)
    print(config_path)
    print('=' * 80)
    print(Path(config_path).read_text(encoding='utf-8'))

## 6. 诊断运行与参数推荐

本节不按设备名称预设训练轮数，而是先执行最小诊断训练，再根据实测耗时推荐正式训练参数。

推荐流程如下：

1. 读取 `random_window` 配置文件中的完整训练轮数。
2. 执行一次最小 S-DDAE 诊断训练，估算 S-DDAE 单轮耗时。
3. 使用诊断 S-DDAE checkpoint 执行一次最小 SS-DDFAE 诊断训练，估算 SS-DDFAE 单轮耗时。
4. 根据目标训练总时长计算缩放系数。
5. 输出本次正式训练推荐参数。

默认 `TARGET_TOTAL_MINUTES = 40`。当前推荐配置通常能在 Mac 上直接完整运行；若只做快速演示，可调小该值。

In [ ]:
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import torch
import yaml

TARGET_TOTAL_MINUTES = 40
PREFER_MPS = True
SEED = 1
LABEL_RATIO = 1.0
WINDOW_SIZE = 16
QUALITY_DELAY = 0
LATENT_DIM = 32
DIAG_BASELINE_NAME = 'diagnose_runtime_sddae_probe'
DIAG_IMPROVED_NAME = 'diagnose_runtime_ss_ddfae_probe'
MAC_SDDAE_CONFIG = 'artifacts/runtime_configs/mac_sddae_random_window.yaml'
MAC_SS_CONFIG = 'artifacts/runtime_configs/mac_ss_ddfae_random_window.yaml'
MAC_MULTISCALE_CONFIG = 'artifacts/runtime_configs/mac_sddae.yaml'

Path('.matplotlib-cache').mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str((Path.cwd() / '.matplotlib-cache').resolve())
PYTHON = sys.executable

def load_yaml(path):
    with open(path, 'r', encoding='utf-8') as file:
        return yaml.safe_load(file)

def detect_runtime():
    mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    training_device = 'mps' if PREFER_MPS and mps_available else 'cpu'
    runtime = {
        'python': PYTHON,
        'platform': platform.platform(),
        'training_device': training_device,
        'inference_device': 'cpu',
        'device_name': 'Apple MPS' if training_device == 'mps' else 'CPU',
        'cuda_available': torch.cuda.is_available(),
        'mps_available': mps_available,
        'gpu_memory_gb': 0.0,
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        runtime['device_name'] = props.name
        runtime['gpu_memory_gb'] = round(props.total_memory / 1024**3, 2)
    return runtime

def write_mac_config(source_path, target_path, device):
    config = load_yaml(source_path)
    config['device'] = device
    Path(target_path).parent.mkdir(parents=True, exist_ok=True)
    with open(target_path, 'w', encoding='utf-8') as file:
        yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)
    return target_path

def run_timed(command):
    print('执行命令:', ' '.join(str(item) for item in command))
    started = time.perf_counter()
    completed = subprocess.run(command, check=True, text=True, capture_output=True)
    elapsed = time.perf_counter() - started
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return elapsed

def remove_diagnostic_artifacts(*names):
    for name in names:
        for path in Path('artifacts').glob(f'**/{name}*'):
            if path.is_file():
                path.unlink()

sddae_config = load_yaml('configs/sddae_random_window.yaml')
ss_config = load_yaml('configs/ss_ddfae_random_window.yaml')
default_pretrain_epochs = int(sddae_config['training']['pretrain_epochs'])
default_finetune_epochs = int(sddae_config['training']['finetune_epochs'])
default_ss_epochs = int(ss_config['training']['epochs'])

runtime = detect_runtime()
TRAINING_DEVICE = runtime['training_device']
INFERENCE_DEVICE = runtime['inference_device']
write_mac_config('configs/sddae_random_window.yaml', MAC_SDDAE_CONFIG, TRAINING_DEVICE)
write_mac_config('configs/ss_ddfae_random_window.yaml', MAC_SS_CONFIG, TRAINING_DEVICE)
write_mac_config('configs/sddae.yaml', MAC_MULTISCALE_CONFIG, TRAINING_DEVICE)

sddae_probe_seconds = run_timed([
    PYTHON, '-m', 'deep_quality.cli.train_sddae',
    '--config', MAC_SDDAE_CONFIG,
    '--seed', str(SEED),
    '--label-ratio', str(LABEL_RATIO),
    '--window-size', str(WINDOW_SIZE),
    '--quality-delay', str(QUALITY_DELAY),
    '--latent-dim', str(LATENT_DIM),
    '--pretrain-epochs', '1',
    '--finetune-epochs', '1',
    '--output-name', DIAG_BASELINE_NAME,
])

ss_probe_seconds = run_timed([
    PYTHON, '-m', 'deep_quality.cli.train_ss_ddfae',
    '--config', MAC_SS_CONFIG,
    '--baseline-checkpoint', f'artifacts/checkpoints/{DIAG_BASELINE_NAME}.pt',
    '--seed', str(SEED),
    '--label-ratio', str(LABEL_RATIO),
    '--window-size', str(WINDOW_SIZE),
    '--quality-delay', str(QUALITY_DELAY),
    '--latent-dim', str(LATENT_DIM),
    '--epochs', '1',
    '--output-name', DIAG_IMPROVED_NAME,
])
remove_diagnostic_artifacts(DIAG_BASELINE_NAME, DIAG_IMPROVED_NAME)

sddae_epoch_seconds = sddae_probe_seconds / 2
ss_epoch_seconds = ss_probe_seconds
full_estimated_seconds = (
    (default_pretrain_epochs + default_finetune_epochs) * sddae_epoch_seconds
    + default_ss_epochs * ss_epoch_seconds
)
target_seconds = TARGET_TOTAL_MINUTES * 60
scale = min(1.0, target_seconds / full_estimated_seconds)

PRETRAIN_EPOCHS = max(1, round(default_pretrain_epochs * scale))
FINETUNE_EPOCHS = max(1, round(default_finetune_epochs * scale))
SS_EPOCHS = max(1, round(default_ss_epochs * scale))

estimated_selected_seconds = (
    (PRETRAIN_EPOCHS + FINETUNE_EPOCHS) * sddae_epoch_seconds
    + SS_EPOCHS * ss_epoch_seconds
)

BASELINE_NAME = f'sddae_r_L{WINDOW_SIZE}_d{QUALITY_DELAY}_z{LATENT_DIM}_r{LABEL_RATIO:g}_rw_nw'
IMPROVED_NAME = f'ss_ddfae_L{WINDOW_SIZE}_d{QUALITY_DELAY}_z{LATENT_DIM}_r{LABEL_RATIO:g}_rw_nw'

diagnosis = {
    **runtime,
    'target_total_minutes': TARGET_TOTAL_MINUTES,
    'prefer_mps': PREFER_MPS,
    'seed': SEED,
    'sddae_probe_seconds': round(sddae_probe_seconds, 3),
    'ss_probe_seconds': round(ss_probe_seconds, 3),
    'sddae_epoch_seconds': round(sddae_epoch_seconds, 3),
    'ss_epoch_seconds': round(ss_epoch_seconds, 3),
    'full_estimated_minutes': round(full_estimated_seconds / 60, 2),
    'recommended_estimated_minutes': round(estimated_selected_seconds / 60, 2),
    'scale': round(scale, 3),
    'label_ratio': LABEL_RATIO,
    'window_size': WINDOW_SIZE,
    'quality_delay': QUALITY_DELAY,
    'latent_dim': LATENT_DIM,
    'split_method': 'random_window',
    'correlation_weight_mode': 'none',
    'pretrain_epochs': PRETRAIN_EPOCHS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'ss_epochs': SS_EPOCHS,
}

display(pd.DataFrame([diagnosis]))
print('S-DDAE 输出名称:', BASELINE_NAME)
print('SS-DDFAE 输出名称:', IMPROVED_NAME)

## 7. 训练 S-DDAE 基线模型

S-DDAE 先通过重构任务学习动态窗口表示，再使用有标签样本微调质量变量预测头。训练轮数和标签比例由上一节诊断结果推荐。

In [ ]:
run_timed([
    PYTHON, '-m', 'deep_quality.cli.train_sddae',
    '--config', MAC_SDDAE_CONFIG,
    '--seed', str(SEED),
    '--label-ratio', str(LABEL_RATIO),
    '--window-size', str(WINDOW_SIZE),
    '--quality-delay', str(QUALITY_DELAY),
    '--latent-dim', str(LATENT_DIM),
    '--pretrain-epochs', str(PRETRAIN_EPOCHS),
    '--finetune-epochs', str(FINETUNE_EPOCHS),
    '--output-name', BASELINE_NAME,
])

## 8. 训练 SS-DDFAE 半监督模型

SS-DDFAE 读取单尺度 S-DDAE checkpoint，迁移兼容参数，然后同时使用有标签样本和无标签样本训练。核心机制包括多层特征预测、注意力融合、一致性约束和伪标签。

In [ ]:
run_timed([
    PYTHON, '-m', 'deep_quality.cli.train_ss_ddfae',
    '--config', MAC_SS_CONFIG,
    '--baseline-checkpoint', f'artifacts/checkpoints/{BASELINE_NAME}.pt',
    '--seed', str(SEED),
    '--label-ratio', str(LABEL_RATIO),
    '--window-size', str(WINDOW_SIZE),
    '--quality-delay', str(QUALITY_DELAY),
    '--latent-dim', str(LATENT_DIM),
    '--epochs', str(SS_EPOCHS),
    '--output-name', IMPROVED_NAME,
])

## 9. 跳过时间序列后处理

`random_window` 会打乱窗口样本顺序，因此 EMA、AR 和在线单步后处理不再具有时间序列含义。主线实验直接使用训练输出的 raw 测试集指标。

In [ ]:
print('random_window 评估不执行 EMA/AR 后处理。')

## 10. 读取 checkpoint 指标

训练命令已经在 `artifacts/tables/*_metrics.json` 中保存 raw 测试集 RMSE、MAE 和 R2。

In [ ]:
for name in [BASELINE_NAME, IMPROVED_NAME]:
    metrics_path = Path(f'artifacts/tables/{name}_metrics.json')
    print()
    print('=' * 80)
    print(metrics_path)
    print('=' * 80)
    print(metrics_path.read_text(encoding='utf-8'))

## 11. 推理耗时说明

本笔记本主线使用随机窗口测试集，不做按时间顺序的在线模拟。若需要严格在线场景，应切回 `split_method: chronological`，并单独报告时间顺序测试结果。

In [ ]:
print('random_window 主线不执行在线单步推理模拟。')

## 12. 汇总 raw 测试指标

重点看两个模型的 raw RMSE、MAE、R2。当前配置的目标是验证 S-DDAE 与 SS-DDFAE 在同分布随机窗口样本上的测试 R2 是否超过 0.95。

In [ ]:
import json
import pandas as pd

def load_json(path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)

rows = []
for name in [BASELINE_NAME, IMPROVED_NAME]:
    train_metrics = load_json(f'artifacts/tables/{name}_metrics.json')
    rows.append({
        'model': train_metrics['model'],
        'label_ratio': train_metrics['label_ratio'],
        'window_size': train_metrics['window_size'],
        'quality_delay': train_metrics['quality_delay'],
        'rmse': train_metrics['rmse'],
        'mae': train_metrics['mae'],
        'r2': train_metrics['r2'],
    })

summary_df = pd.DataFrame(rows)
baseline_rmse = float(summary_df.loc[summary_df['model'] == 'sddae_r', 'rmse'].iloc[0])
baseline_mae = float(summary_df.loc[summary_df['model'] == 'sddae_r', 'mae'].iloc[0])
baseline_r2 = float(summary_df.loc[summary_df['model'] == 'sddae_r', 'r2'].iloc[0])
summary_df['rmse_vs_sddae'] = summary_df['rmse'] - baseline_rmse
summary_df['mae_vs_sddae'] = summary_df['mae'] - baseline_mae
summary_df['r2_vs_sddae'] = summary_df['r2'] - baseline_r2
display(summary_df)

best_row = summary_df.sort_values(['r2', 'rmse'], ascending=[False, True]).iloc[0]
print('R2 最高模型:', best_row['model'])
print('最高 R2:', best_row['r2'])

## 13. 展示预测曲线、散点图和残差图

预测曲线用于展示模型能否跟踪质量变量变化；散点图用于展示预测值和真实值的一致性；残差图用于观察误差是否存在明显偏移或趋势。

In [ ]:
from IPython.display import Image, display

for name in [BASELINE_NAME, IMPROVED_NAME]:
    print()
    print('=' * 80)
    print(name)
    print('=' * 80)
    for suffix in ['prediction', 'scatter', 'residuals']:
        path = Path(f'artifacts/figures/{name}_{suffix}.png')
        if path.exists():
            print(path)
            display(Image(filename=str(path)))

## 14. 查看输出文件清单

所有训练产物都在 `artifacts/` 下。展示时优先打开 `tables` 中的指标文件和 `figures` 中的图像文件。

In [ ]:
for path in sorted(Path('artifacts').glob('**/*')):
    if path.is_file():
        print(path.as_posix())

## 15. 可选：多尺度 S-DDAE 展示

多尺度 S-DDAE 同时使用不同窗口长度和采样步长建模动态信息。SS-DDFAE 当前只支持单尺度基线 checkpoint，因此多尺度部分作为 S-DDAE 扩展实验单独展示。

In [ ]:
MULTISCALE_NAME = 'sddae_r_ms40x1-24x2-12x4_d12_z52_r1'

run_timed([
    PYTHON, '-m', 'deep_quality.cli.train_sddae',
    '--config', MAC_MULTISCALE_CONFIG,
    '--seed', str(SEED),
    '--label-ratio', '1.0',
    '--quality-delay', '12',
    '--latent-dim', '52',
    '--scales', '40x1,24x2,12x4',
    '--output-name', MULTISCALE_NAME,
])